## Jupyter Notebook Data Science Migration Pipeline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("[-] Loading structural baseline from real PaySim dataset...")
df = pd.read_csv('paysim.csv', nrows=300000)

## Data Preprocessing, Feature Engineering, and Transformation

In [ ]:
column_mapping = {
    'step': 'Time_Step', 'type': 'Transaction_Type', 'amount': 'Amount',
    'oldbalanceOrg': 'Old_Balance_Orig', 'newbalanceOrig': 'New_Balance_Orig',
    'oldbalanceDest': 'Old_Balance_Dest', 'newbalanceDest': 'New_Balance_Dest',
    'isFraud': 'Is_Fraud'
}

df_renamed = df.rename(columns=column_mapping)
df_cleaned = df_renamed.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'], errors='ignore')
df_cleaned = pd.get_dummies(df_cleaned, columns=['Transaction_Type'], drop_first=True)
df_cleaned = df_cleaned.dropna()

X = df_cleaned.drop('Is_Fraud', axis=1)
y = df_cleaned['Is_Fraud']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"[+] Pipeline preprocessing completed. Input dataset tensor shape: {X.shape}")

## Exploratory Data Analysis (EDA) Mapping

In [ ]:
%matplotlib inline
print("[-] Generating Feature Linear Correlation Heatmap Matrix...")
plt.figure(figsize=(12, 8))
sns.heatmap(df_cleaned.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title("Exploratory Data Analysis: Correlation Heatmap Matrix")
plt.tight_layout()
plt.show()

## Statistical Evaluation, Inference, and Matrix Visualization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.50, random_state=42, stratify=y
)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(32, 16), activation='relu', solver='adam', 
    max_iter=50, random_state=42, verbose=True
)

print("[-] Commencing stochastic training routines on Multi-Layer Perceptron...")
mlp_model.fit(X_train, y_train)

print("\n[-] Extracting class classification probabilities on unseen validation samples...")
y_pred = mlp_model.predict(X_test)

print("\n--- EVALUATION METRICS REPORT ---")
print(classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Fraud (1)']))

conf_matrix = confusion_matrix(y_test, y_pred)
visual_display = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Legitimate (0)', 'Fraud (1)'])
visual_display.plot(cmap='Reds')
plt.title("Confusion Matrix Validation - Deep MLP Neural Network")
plt.show()